<a href="https://colab.research.google.com/github/soumaya-lamrhari/customer-conversion-prediction/blob/main/notebooks/2_data_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preprocessing

## Objective

In this notebook, I prepare the Bank Marketing dataset for machine learning based on the findings from the exploratory data analysis.

The main steps include handling missing values, addressing data leakage, preparing categorical and numerical features, and splitting the data into training and test sets.

In [1]:
!pip install -q ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

# Load the original Bank Marketing dataset
bank_marketing = fetch_ucirepo(id=222)

X = bank_marketing.data.features.copy()
y = bank_marketing.data.targets.copy()

In [2]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (45211, 16)
Target shape: (45211, 1)


In [3]:
# Remove duration to avoid data leakage since it is only known after contact
X = X.drop(columns=["duration"])

print("Features shape after removing duration:", X.shape)

Features shape after removing duration: (45211, 15)


In [4]:
# Missing poutcome mostly means there was no previous campaign contact
X["poutcome"] = X["poutcome"].fillna("no_previous_contact")

X["poutcome"].value_counts()

,count
poutcome,
no_previous_contact,36959
failure,4901
other,1840
success,1511


In [5]:
# Keep missing contact method as its own category
X["contact"] = X["contact"].fillna("unknown")

X["contact"].value_counts()

,count
contact,
cellular,29285
unknown,13020
telephone,2906


In [6]:
# Keep missing job and education values as unknown categories
X["job"] = X["job"].fillna("unknown")
X["education"] = X["education"].fillna("unknown")

X[["job", "education"]].isna().sum()

,0
job,0
education,0


In [7]:
# Check for any remaining missing values
X.isna().sum()

,0
age,0
job,0
marital,0
education,0
default,0
balance,0
housing,0
loan,0
contact,0
day_of_week,0


In [8]:
# Convert the target to binary values for modeling
y = y["y"].map({"yes": 1, "no": 0})

y.value_counts()

,count
y,
0,39922
1,5289


In [9]:
# Rename the column to reflect what the values represent
X = X.rename(columns={"day_of_week": "day_of_month"})

X.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day_of_month', 'month', 'campaign', 'pdays',
       'previous', 'poutcome'],
      dtype='object')

In [10]:
from sklearn.model_selection import train_test_split

# Keep the same conversion distribution in the training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (36168, 15)
Test set: (9043, 15)


In [11]:
# Separate numerical and categorical features for preprocessing
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

Numerical features: ['age', 'balance', 'day_of_month', 'campaign', 'pdays', 'previous']
Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# One-hot encode categorical features and keep numerical features unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome']),
                                ('num', 'passthrough',
                                 ['age', 'balance', 'day_of_month', 'campaign',
                                  'pdays', 'previous'])])

In [13]:
# Learn the categories from the training data and transform it
X_train_processed = preprocessor.fit_transform(X_train)

print("Before preprocessing:", X_train.shape)
print("After preprocessing:", X_train_processed.shape)

Before preprocessing: (36168, 15)
After preprocessing: (36168, 50)


In [14]:
feature_names = preprocessor.get_feature_names_out()

print(feature_names)
print("Total features:", len(feature_names))

['cat__job_admin.' 'cat__job_blue-collar' 'cat__job_entrepreneur'
 'cat__job_housemaid' 'cat__job_management' 'cat__job_retired'
 'cat__job_self-employed' 'cat__job_services' 'cat__job_student'
 'cat__job_technician' 'cat__job_unemployed' 'cat__job_unknown'
 'cat__marital_divorced' 'cat__marital_married' 'cat__marital_single'
 'cat__education_primary' 'cat__education_secondary'
 'cat__education_tertiary' 'cat__education_unknown' 'cat__default_no'
 'cat__default_yes' 'cat__housing_no' 'cat__housing_yes' 'cat__loan_no'
 'cat__loan_yes' 'cat__contact_cellular' 'cat__contact_telephone'
 'cat__contact_unknown' 'cat__month_apr' 'cat__month_aug' 'cat__month_dec'
 'cat__month_feb' 'cat__month_jan' 'cat__month_jul' 'cat__month_jun'
 'cat__month_mar' 'cat__month_may' 'cat__month_nov' 'cat__month_oct'
 'cat__month_sep' 'cat__poutcome_failure'
 'cat__poutcome_no_previous_contact' 'cat__poutcome_other'
 'cat__poutcome_success' 'num__age' 'num__balance' 'num__day_of_month'
 'num__campaign' 'num__pda

In [15]:
# Apply the preprocessing learned from the training data to the test set
X_test_processed = preprocessor.transform(X_test)

print("Training data:", X_train_processed.shape)
print("Test data:", X_test_processed.shape)

Training data: (36168, 50)
Test data: (9043, 50)


In [16]:
# Check that both sets have a similar conversion rate
print("Training conversion rate:", round(y_train.mean() * 100, 2), "%")
print("Test conversion rate:", round(y_test.mean() * 100, 2), "%")

Training conversion rate: 11.7 %
Test conversion rate: 11.7 %


## Preprocessing Summary

- Removed `duration` to prevent data leakage.
- Kept meaningful missing information as separate categories instead of dropping customers.
- Converted the target from `yes`/`no` to `1`/`0`.
- Renamed `day_of_week` to `day_of_month` to reflect the actual values.
- Split the data into 80% training and 20% testing using stratification.
- One-hot encoded categorical features while keeping numerical features unchanged.
- Fitted the preprocessing steps on the training data only and applied the same transformations to the test data.
- The final processed dataset contains 50 features.